# Random Forest — Ensemble Baseline for Exoplanet Detection

**Random Forest on standardized 39 V4 features with nested CV hyperparameter search. The strongest classical baseline in our preliminary tests; included as the primary tree-ensemble reference.**

Uses the same 39 V4 features and identical OOF protocol (10-fold StratifiedKFold x 5 reps, seeds 42-46) with nested CV (RandomizedSearchCV, 3-fold inner CV, 15 iterations) as all other classical ML notebooks. Results are directly comparable across all models.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, accuracy_score
)
import matplotlib.pyplot as plt

# ── Load observations ──
observations = pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl')
print(f"Total observations: {len(observations)}")
print(f"Stars: {observations['star_name'].nunique()}")

## 1. Compute base physical features (18 features, same as RF baseline)

In [ ]:
def compute_star_features(group):
    """Compute summary statistics for one star from its observation sequence."""
    features = {}

    # -- RV statistics --
    features['rv_std'] = group['rv_centered'].std()
    features['rv_range'] = group['rv_centered'].max() - group['rv_centered'].min()
    features['rv_mean_abs_dev'] = (group['rv_centered'] - group['rv_centered'].median()).abs().mean()
    features['rv_skew'] = group['rv_centered'].skew()
    features['rv_kurtosis'] = group['rv_centered'].kurtosis()

    # -- RV error statistics --
    features['rv_err_mean'] = group['rv_err'].mean()
    features['rv_err_std'] = group['rv_err'].std()

    # -- Activity indicator statistics --
    features['rhkp_std'] = group['RHKp'].std()
    features['rhkp_range'] = group['RHKp'].max() - group['RHKp'].min()
    features['rhkp_mean'] = group['RHKp'].mean()
    features['halpha_std'] = group['Halpha'].std()
    features['halpha_range'] = group['Halpha'].max() - group['Halpha'].min()
    features['halpha_mean'] = group['Halpha'].mean()

    # -- RV-activity correlations --
    features['rv_rhkp_corr'] = group['rv_centered'].corr(group['RHKp'])
    features['rv_halpha_corr'] = group['rv_centered'].corr(group['Halpha'])
    features['rhkp_halpha_corr'] = group['RHKp'].corr(group['Halpha'])

    # -- Binary variance flags: 1 if indicator had variance, 0 if constant (corr NaN -> aliased to 0) --
    features['rhkp_has_variance'] = 1 if group['RHKp'].std() > 0 else 0
    features['halpha_has_variance'] = 1 if group['Halpha'].std() > 0 else 0

    # -- Label --
    features['has_exoplanets'] = group['has_exoplanets'].iloc[0]

    return pd.Series(features)

star_features = observations.groupby('star_name').apply(compute_star_features, include_groups=False).reset_index()
star_features = star_features.fillna(0)
star_features['has_exoplanets'] = star_features['has_exoplanets'].astype(int)

physical_features = [
    'rv_std', 'rv_range', 'rv_mean_abs_dev', 'rv_skew', 'rv_kurtosis',
    'rv_err_mean', 'rv_err_std',
    'rhkp_std', 'rhkp_range', 'rhkp_mean',
    'halpha_std', 'halpha_range', 'halpha_mean',
    'rv_rhkp_corr', 'rv_halpha_corr', 'rhkp_halpha_corr',
    'rhkp_has_variance', 'halpha_has_variance'
]

print(f"Stars: {len(star_features)}")
print(f"Positive: {star_features['has_exoplanets'].sum()}")
print(f"Negative: {(star_features['has_exoplanets'] == 0).sum()}")
print(f"Features ({len(physical_features)}): {physical_features}")

## 2. Compute V2 derived features (24 features: 5 enhanced + 16 LS + 3 uncertainty)

Identical to `rf_multiseed.ipynb` cell 14 — same `compute_v2_features` function.

In [ ]:
# Optimize v2: prune dead features + add targeted new features + seed-45 regime
# Run this inside the notebook OR as a standalone script after the existing cells
print("=" * 72)
print("PHYS-NEW v2: Prune + extend + seed-45 regime")
print("=" * 72)

import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)
from astropy.timeseries import LombScargle
warnings.filterwarnings('ignore')

N_FREQ = 500
F_MIN_DAYS = 1.0
LS_METHOD = 'fast'

# ── Features to KEEP after pruning ──
# Dropped: halpha_has_variance, rhkp_has_variance (0.0000 importance, always 1)
# Dropped: rv_rhkp_corr, rv_halpha_corr (signed corrs, dominated by absolute versions)
CORE_LS = ['rv_ls_peak_power', 'rv_ls_log_peak_period',
           'rv_ls_mean_power', 'rhkp_ls_peak_power', 'rhkp_rv_power_ratio',
           'halpha_ls_peak_power', 'halpha_rv_power_ratio', 'min_activity_rv_ratio']

MULTI_LS = ['rv_top2_power_ratio', 'rv_top3_power_ratio',
            'rv_power_conc_top10pct', 'rhkp_power_conc_top10pct',
            'halpha_top2_power_ratio']

# NEW v2 features (3 more, extending the uncertainty + periodogram directions)
V2_LS = ['rv_power_conc_top5pct', 'rv_power_conc_top1pct',
         'rv_periodic_snr']

ALL_LS = CORE_LS + MULTI_LS + V2_LS  # 8 + 5 + 3 = 16

ENH = ['rv_rhkp_corr_abs', 'rv_halpha_corr_abs', 'rv_rhkp_partial_corr',
       'rv_rhkp_corr_std', 'rhkp_rv_ratio']

UNCERT = ['rv_excess_std', 'rv_snr', 'rv_weighted_amplitude']

DERIVED = ENH + ALL_LS + UNCERT  # 5 + 16 + 3 = 24


def compute_v2_features(group):
    """Derived features for one star: 5 enh + 16 LS + 3 uncert + 3 v2 = 27 total."""
    f = {}
    rv     = group['rv_centered'].values.astype(float)
    rhkp   = group['RHKp'].values.astype(float)
    halpha = group['Halpha'].values.astype(float)
    bjd    = group['bjd'].values.astype(float)
    rv_err = group['rv_err'].values.astype(float)
    n = len(rv)

    rv_s   = float(np.std(rv))     if n >= 2 else 0.0
    rv_s   = rv_s if rv_s == rv_s else 0.0
    rhkp_s = float(np.std(rhkp))   if n >= 2 else 0.0
    rhkp_s = rhkp_s if rhkp_s == rhkp_s else 0.0
    ha_s   = float(np.std(halpha)) if n >= 2 else 0.0
    ha_s   = ha_s if ha_s == ha_s else 0.0

    # ── Uncertainty (3) ──
    rv_var = rv_s ** 2
    mean_rv_err2 = float(np.mean(rv_err ** 2)) if n > 0 else 0.0
    excess_var = max(rv_var - mean_rv_err2, 0.0)
    f['rv_excess_std']  = float(np.sqrt(excess_var))
    f['rv_snr']         = rv_s / float(np.mean(rv_err)) if n > 0 and float(np.mean(rv_err)) > 0 else 0.0
    f['rv_weighted_amplitude'] = rv_s  # fallback

    # ── Enhanced (5) ──
    f['rv_rhkp_corr_abs']   = abs(float(np.corrcoef(rv, rhkp)[0, 1])) if (rv_s > 0 and rhkp_s > 0) else 0.0
    f['rv_halpha_corr_abs'] = abs(float(np.corrcoef(rv, halpha)[0, 1])) if (rv_s > 0 and ha_s > 0) else 0.0
    if n > 2 and rv_s > 0 and rhkp_s > 0:
        t = bjd - bjd.mean()
        A = np.column_stack([t, np.ones_like(t)])
        def _resid(y):
            coef = np.linalg.lstsq(A, y, rcond=None)[0]
            return y - (coef[0] * t + coef[1])
        rv_r, rhkp_r = _resid(rv), _resid(rhkp)
        std_rv_r, std_rhkp_r = float(np.std(rv_r)), float(np.std(rhkp_r))
        f['rv_rhkp_partial_corr'] = float(np.corrcoef(rv_r, rhkp_r)[0, 1]) if (std_rv_r > 0 and std_rhkp_r > 0) else 0.0
    else:
        f['rv_rhkp_partial_corr'] = 0.0
    w = min(20, n // 2)
    if w >= 5 and rv_s > 0 and rhkp_s > 0:
        step, corrs = max(1, w // 2), []
        for s in range(0, n - w + 1, step):
            if np.std(rhkp[s:s + w]) > 0 and np.std(rv[s:s + w]) > 0:
                c = float(np.corrcoef(rv[s:s + w], rhkp[s:s + w])[0, 1])
                if c == c and not np.isnan(c):
                    corrs.append(c)
        f['rv_rhkp_corr_std'] = float(np.std(corrs)) if len(corrs) >= 2 else 0.0
    else:
        f['rv_rhkp_corr_std'] = 0.0
    f['rhkp_rv_ratio'] = rhkp_s / rv_s if rv_s > 0 else 0.0

    # ── Lomb-Scargle (16 LS + 3 v2 = 19 total from LS) ──
    baseline = float(bjd.max() - bjd.min()) if n > 1 else 0.0
    p_max = max(baseline, 2.0) if baseline > 0 else 2.0
    freqs = np.linspace(1.0 / p_max, 1.0 / F_MIN_DAYS, N_FREQ)
    distinct_t = np.unique(np.round(bjd, 6)).size if n > 0 else 0
    can_run_ls = (n >= 4) and (rv_s > 0) and (baseline > 0) and (distinct_t >= 4)

    # Default zero for all LS + v2 keys
    for k in ALL_LS:
        f[k] = 0.0

    if can_run_ls:
        try:
            ls_rv = LombScargle(bjd, rv, dy=rv_err, normalization='standard', fit_mean=True)
            p_rv = ls_rv.power(freqs, method=LS_METHOD)
            pi = int(np.argmax(p_rv))
            rv_peak_power = float(p_rv[pi])
            rv_peak_period = float(1.0 / freqs[pi]) if freqs[pi] > 0 else 0.0

            # Weighted amplitude
            x_fit = 2.0 * np.pi * freqs[pi] * (bjd - bjd.min())
            A_fit = np.column_stack([np.sin(x_fit), np.cos(x_fit)])
            coef = np.linalg.lstsq(A_fit, rv, rcond=None)[0]
            fitted_amp = float(np.sqrt(coef[0]**2 + coef[1]**2))
            f['rv_weighted_amplitude'] = fitted_amp

            # v2: periodic_snr = fitted amplitude / mean rv_err
            mean_err = float(np.mean(rv_err)) if n > 0 else 0.0
            f['rv_periodic_snr'] = fitted_amp / mean_err if mean_err > 0 else 0.0

            ls_rhk = LombScargle(bjd, rhkp, normalization='standard', fit_mean=True)
            ls_hal = LombScargle(bjd, halpha, normalization='standard', fit_mean=True)
            p_rhk = ls_rhk.power(freqs, method=LS_METHOD)
            p_hal = ls_hal.power(freqs, method=LS_METHOD)

            rhkp_pow_at_rv = float(p_rhk[pi])
            halpha_pow_at_rv = float(p_hal[pi])
            rhkp_ratio   = rhkp_pow_at_rv / (rv_peak_power + 1e-9)
            halpha_ratio = halpha_pow_at_rv / (rv_peak_power + 1e-9)

            # Multi-period: peak ratios
            p_sorted_idx = np.argsort(p_rv)[::-1]
            top2 = float(p_rv[p_sorted_idx[1]] / p_rv[p_sorted_idx[0]]) if len(p_sorted_idx) > 1 and p_rv[p_sorted_idx[0]] > 0 else 0.0
            top3 = float(p_rv[p_sorted_idx[2]] / p_rv[p_sorted_idx[0]]) if len(p_sorted_idx) > 2 and p_rv[p_sorted_idx[0]] > 0 else 0.0

            # Power concentration at multiple thresholds
            p_sorted = np.sort(p_rv)
            total_p = max(np.sum(p_rv), 1e-9)
            conc_10  = float(np.sum(p_sorted[-(max(1, len(p_rv)//10)):]) / total_p)
            conc_5   = float(np.sum(p_sorted[-(max(1, len(p_rv)//20)):]) / total_p)
            conc_1   = float(np.sum(p_sorted[-(max(1, len(p_rv)//100)):]) / total_p)

            # RHKp concentration
            p_rhk_sorted = np.sort(p_rhk)
            rhk_conc_10 = float(np.sum(p_rhk_sorted[-(max(1, len(p_rhk)//10)):]) / max(np.sum(p_rhk), 1e-9))

            # Halpha top2
            p_hal_sorted_idx = np.argsort(p_hal)[::-1]
            hal_top2 = float(p_hal[p_hal_sorted_idx[1]] / p_hal[p_hal_sorted_idx[0]]) if len(p_hal_sorted_idx) > 1 and p_hal[p_hal_sorted_idx[0]] > 0 else 0.0

            # Store LS features
            f['rv_ls_peak_power']        = rv_peak_power
            f['rv_ls_log_peak_period']   = float(np.log10(max(rv_peak_period, 1e-9)))
            f['rv_ls_mean_power']        = float(np.mean(p_rv))
            f['rhkp_ls_peak_power']      = float(np.max(p_rhk))
            f['rhkp_rv_power_ratio']     = rhkp_ratio
            f['halpha_ls_peak_power']    = float(np.max(p_hal))
            f['halpha_rv_power_ratio']   = halpha_ratio
            f['min_activity_rv_ratio']   = min(rhkp_ratio, halpha_ratio)
            f['rv_top2_power_ratio']     = top2
            f['rv_top3_power_ratio']     = top3
            f['rv_power_conc_top10pct']  = conc_10
            f['rhkp_power_conc_top10pct'] = rhk_conc_10
            f['halpha_top2_power_ratio'] = hal_top2
            # v2 features
            f['rv_power_conc_top5pct']   = conc_5
            f['rv_power_conc_top1pct']   = conc_1
        except Exception:
            pass  # already zeroed

    f['has_exoplanets'] = int(group['has_exoplanets'].iloc[0])
    return pd.Series(f)


# ── Compute ──
v2_derived = (observations.groupby('star_name', sort=True)
              .apply(compute_v2_features, include_groups=False)
              .reset_index()
              .fillna(0))

opt2 = star_features.merge(v2_derived[['star_name'] + DERIVED],
                           on='star_name', how='left').fillna(0)

# Define physical features (original 18 minus 4 dead ones)
PHYS_CLEAN = [f for f in physical_features
              if f not in ('halpha_has_variance', 'rhkp_has_variance',
                           'rv_rhkp_corr', 'rv_halpha_corr')]
PHYS_V2 = PHYS_CLEAN + DERIVED  # 14 physical + 24 derived = 38

## 3. Build V4 feature set (39 features)

V2 (38) + `cadence_median` (1) = 39. Identical to `rf_multiseed.ipynb` cell 18.

In [ ]:

# ── Build V4 feature set: v2 (38) + cadence_median (1) = 39 ──
def compute_cadence(group):
    bjd = group['bjd'].values.astype(float)
    if len(bjd) >= 2:
        dbjd = np.diff(np.sort(bjd))
        return float(np.median(dbjd))
    return 0.0

cadence = (observations.groupby('star_name', sort=True)
           .apply(compute_cadence, include_groups=False)
           .reset_index()
           .rename(columns={0: 'cadence_median'})
           .fillna(0))

opt4 = opt2.merge(cadence[['star_name', 'cadence_median']],
                  on='star_name', how='left').fillna(0)

PHYS_V4 = PHYS_V2 + ['cadence_median']  # 38 + 1 = 39

print(f"PHYS_V4: {len(PHYS_V4)} features (v2 38 + cadence_median)")
print(f"Stars: {len(opt4)}, pos={int(opt4.has_exoplanets.sum())}, "
      f"neg={int((opt4.has_exoplanets == 0).sum())}")


## 4. OOF Evaluation — Random Forest with Nested CV Hyperparameter Search

10-fold StratifiedKFold x 5 repetitions (seeds 42-46). Nested CV: RandomizedSearchCV with 3-fold inner CV per outer fold, scoring on PR-AUC. Every star gets a prediction from a model that didn't see it.

In [ ]:
# ── Model configuration ──
MODEL_NAME = 'Random Forest'

# ── Hyperparameter search space (nested CV within each OOF fold) ──
PARAM_DISTRIBUTIONS = {
    'n_estimators': [500, 1000, 2000, 3000],
    'max_depth': [10, 20, 30, 50, None],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None],
}
N_INNER_FOLDS = 3
N_ITER = 15
USE_STANDARDIZATION = True
SAVE_NAME = '/kaggle/working/rf_oof_results.pkl'

# ── OOF Protocol (identical to all classical ML: 10-fold StratifiedKFold x 5 reps) ──
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)
warnings.filterwarnings('ignore')

N_REPS = 5
N_FOLDS = 10

X = opt4[PHYS_V4].values
y = opt4['has_exoplanets'].values.astype(int)
n_stars = len(y)

print(f"OOF Protocol: {N_FOLDS}-fold StratifiedKFold x {N_REPS} repetitions")
print(f"Model: {MODEL_NAME}")
print(f"Features: {len(PHYS_V4)}")
print(f"Stars: {n_stars}, pos={y.sum()}, neg={(1-y).sum()}")

all_oof_probs = np.zeros((n_stars, N_REPS))
rep_metrics = {'rep': [], 'pr_auc': [], 'roc_auc': [],
               'f1': [], 'f05': [], 'precision': [], 'recall': []}

print(f"\n{'=' * 72}")
print(f"OUT-OF-FOLD EVALUATION")
print(f"{'=' * 72}")

for rep in range(N_REPS):
    oof_probs = np.zeros(n_stars)
    seed = 42 + rep

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test = X[test_idx]

        # ── Standardization (fit on train, transform test) ──
        if USE_STANDARDIZATION:
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)
        else:
            X_train_s = X_train
            X_test_s = X_test

        # ── Nested CV hyperparameter search + fit + predict ──
        from sklearn.ensemble import RandomForestClassifier
        base_clf = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=seed)
        inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed)
        search = RandomizedSearchCV(
            base_clf, PARAM_DISTRIBUTIONS, n_iter=N_ITER,
            cv=inner_cv, scoring='average_precision', n_jobs=-1,
            random_state=seed,
        )
        search.fit(X_train_s, y_train)
        fold_probs = search.predict_proba(X_test_s)[:, 1]

        oof_probs[test_idx] = fold_probs

    all_oof_probs[:, rep] = oof_probs

    # Per-rep metrics on OOF predictions
    roc = roc_auc_score(y, oof_probs)
    pr  = average_precision_score(y, oof_probs)
    vp, vr, vt = precision_recall_curve(y, oof_probs)
    vf1 = 2 * vp * vr / (vp + vr + 1e-8)
    bi = int(np.argmax(vf1))
    thr = float(vt[bi]) if bi < len(vt) else 0.5
    preds = (oof_probs >= thr).astype(int)
    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = f1_score(y, preds, zero_division=0)
    f05 = fbeta_score(y, preds, beta=0.5, zero_division=0)

    rep_metrics['rep'].append(rep)
    rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc)
    rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc)
    rep_metrics['recall'].append(rec)

    print(f"  Rep {rep} (seed {seed}): "
          f"PR-AUC={pr:.4f}  ROC-AUC={roc:.4f}  "
          f"F1={f1:.4f}  F0.5={f05:.4f}  "
          f"P={prc:.3f}  R={rec:.3f}")

# ── Summary ──
rep_df = pd.DataFrame(rep_metrics)

print(f"\n{'=' * 72}")
print(f"{MODEL_NAME} OOF SUMMARY: 10-fold x 5 reps, {len(PHYS_V4)} features")
print(f"{'=' * 72}")

print("\nPer-rep PR-AUC:")
for _, row in rep_df.iterrows():
    print(f"  Rep {int(row['rep'])}: {row['pr_auc']:.4f}")

print(f"\nAggregate (n={N_REPS} reps):")
for m_idx in ['pr_auc', 'roc_auc', 'f1', 'f05', 'precision', 'recall']:
    v = rep_df[m_idx].values
    print(f"  {m_idx:11s}: {v.mean():.4f} +/- {v.std(ddof=1):.4f}  "
          f"(min={v.min():.4f}, max={v.max():.4f})")

# ── Combined OOF (average probabilities across reps) ──
combined_y = y
avg_oof = all_oof_probs.mean(axis=1)
combined_pr = average_precision_score(combined_y, avg_oof)
combined_roc = roc_auc_score(combined_y, avg_oof)
vp, vr, vt = precision_recall_curve(combined_y, avg_oof)
vf1 = 2 * vp * vr / (vp + vr + 1e-8)
bi = int(np.argmax(vf1))
thr = float(vt[bi]) if bi < len(vt) else 0.5
combined_prob = avg_oof
combined_f1 = f1_score(y, (avg_oof >= thr).astype(int), zero_division=0)
combined_f05 = fbeta_score(combined_y, (avg_oof >= thr).astype(int), beta=0.5,
                           zero_division=0)
cm = confusion_matrix(combined_y, (avg_oof >= thr).astype(int))
tn, fp, fn, tp = cm.ravel()

print(f"\n{'=' * 72}")
print(f"COMBINED OOF (probabilities averaged across {N_REPS} reps)")
print(f"{'=' * 72}")
print(f"  PR-AUC:  {combined_pr:.4f}")
print(f"  ROC-AUC: {combined_roc:.4f}")
print(f"  F1:      {combined_f1:.4f}  (threshold={thr:.4f})")
print(f"  F0.5:    {combined_f05:.4f}")
print(f"  Confusion: TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"  Precision: {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}  "
      f"Recall: {tp/(tp+fn) if (tp+fn)>0 else 0:.4f}")

# ── Save results ──
results = {
    'model': MODEL_NAME,
    'pr_auc_combined': combined_pr,
    'roc_auc_combined': combined_roc,
    'f1': combined_f1,
    'f05': combined_f05,
    'rep_metrics': rep_df.to_dict('records'),
    'n_features': len(PHYS_V4),
    'n_stars': n_stars,
}
import pickle
with open(SAVE_NAME, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to {SAVE_NAME}")
print(f"\nFINAL: {MODEL_NAME} combined OOF PR-AUC = {combined_pr:.4f}")

# ── Bootstrap 95% CIs for combined OOF metrics ──
from split import bootstrap_roc_auc, bootstrap_pr_auc

print("\nBOOTSTRAP 95% CI (200 resamples):")
print(f"{'=' * 72}")

roc_point, roc_lo, roc_hi = bootstrap_roc_auc(combined_y, combined_prob)
print(f"  ROC-AUC: {roc_point:.4f}  [{roc_lo:.4f}, {roc_hi:.4f}]")

pr_point, pr_lo, pr_hi = bootstrap_pr_auc(combined_y, combined_prob)
print(f"  PR-AUC:  {pr_point:.4f}  [{pr_lo:.4f}, {pr_hi:.4f}]")
